<a href="https://colab.research.google.com/github/PrisceliaAnggani/intel-image-classification-cnn/blob/main/Intel_Image_Classification_using_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Intel Image Classification using CNN

A Convolutional Neural Network (CNN) built with TensorFlow/Keras to classify
natural scenes into 6 categories: buildings, forest, glacier, mountain, sea, and street.

- Dataset: Intel Image Classification (Kaggle)
- Model: Custom CNN with BatchNormalization and Dropout
- Test Accuracy: ~80%

1. Install Dependencies

In [ ]:
!pip install tensorflow keras numpy pandas matplotlib plotly Pillow scikit-learn

2. Import Libraries

In [ ]:
import os
import zipfile
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import classification_report
from collections import Counter
from PIL import Image

# For reproducibility
tf.random.set_seed(42)
np.random.seed(42)

3. Load Dataset

In [ ]:
import kagglehub

# Download dataset directly
path = kagglehub.dataset_download("puneet6060/intel-image-classification")
print("Path to dataset files:", path)

# Set paths
BASE_DIR = path

train_dir = os.path.join(BASE_DIR, 'seg_train', 'seg_train')
test_dir  = os.path.join(BASE_DIR, 'seg_test', 'seg_test')
pred_dir  = os.path.join(BASE_DIR, 'seg_pred', 'seg_pred')

print("Train dir:", train_dir)
print("Test dir:", test_dir)
print("Pred dir:", pred_dir)

4. Data Preprocessing & Augmentation

In [ ]:
# Training data - with augmentation to improve model generalization
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=20,
    zoom_range=0.2,
    validation_split=0.2  # 20% of train data used for validation
)

# Test data - NO augmentation, only rescale
test_datagen = ImageDataGenerator(rescale=1.0/255)

5. Load and Explore Data

In [ ]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='training'  # uses 80% of train folder
)

val_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='validation'  # uses 20% of train folder
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    shuffle=False  # important for accurate evaluation later
)

print("Classes found:", train_data.class_indices)

6. Build CNN Model

In [ ]:
model = Sequential([
    # Block 1
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    # Block 2
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    # Block 3
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    # Classifier
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax')  # 6 classes
])

model.summary()

7. Train Model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks to save best model and stop early if no improvement
callbacks = [
    ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_accuracy'),
    EarlyStopping(patience=5, monitor='val_accuracy', restore_best_weights=True)
]

history = model.fit(
    train_data,
    epochs=20,
    validation_data=val_data,
    callbacks=callbacks
)

8. Visualize Results

In [ ]:
# Plot accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

9. Evaluate Model

In [ ]:
# Evaluate on test data
test_loss, test_accuracy = model.evaluate(test_data)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

# Detailed classification report
predictions = model.predict(test_data)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_data.classes
class_names = list(test_data.class_indices.keys())

print("\nClassification Report:")
print(classification_report(true_classes, predicted_classes, target_names=class_names))

10. Saving and Downloading Model

In [ ]:
# Save the final model
model.save('final_model.keras')
print("Model saved!")

# Download it to your computer
from google.colab import files
files.download('final_model.keras')
files.download('best_model.keras')

In [ ]:
# Check what's inside pred_dir
print("pred_dir:", pred_dir)
print("\nContents:", os.listdir(pred_dir))

11. Predictions

In [ ]:
import random

# Get random images directly from pred folder
all_images = [f for f in os.listdir(pred_dir) if f.endswith('.jpg')]
sample_images = random.sample(all_images, 12)  # pick 12 random ones

plt.figure(figsize=(15, 10))
class_names = list(train_data.class_indices.keys())

for i, img_file in enumerate(sample_images):
    img_path = os.path.join(pred_dir, img_file)

    img = image.load_img(img_path, target_size=(150, 150))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)
    predicted_class = class_names[np.argmax(prediction)]
    confidence = np.max(prediction) * 100

    plt.subplot(3, 4, i + 1)
    plt.imshow(image.load_img(img_path))
    plt.title(f"{predicted_class}\n{confidence:.1f}%", fontsize=9)
    plt.axis('off')

plt.suptitle('Model Predictions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import random

# Get some images from test set which has true labels
class_names = list(test_data.class_indices.keys())
sample_images = []
sample_labels = []

# Pick a few images from each class
for class_name, class_idx in test_data.class_indices.items():
    class_path = os.path.join(test_dir, class_name)
    images = os.listdir(class_path)[:2]  # 2 per class
    for img_file in images:
        sample_images.append(os.path.join(class_path, img_file))
        sample_labels.append(class_idx)

plt.figure(figsize=(15, 10))

for i, (img_path, true_idx) in enumerate(zip(sample_images[:12], sample_labels[:12])):
    img = image.load_img(img_path, target_size=(150, 150))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)
    pred_idx = np.argmax(prediction)
    confidence = np.max(prediction) * 100

    true_label = class_names[true_idx]
    pred_label = class_names[pred_idx]

    # Green title if correct, red if wrong
    color = 'green' if true_idx == pred_idx else 'red'

    plt.subplot(3, 4, i + 1)
    plt.imshow(image.load_img(img_path))
    plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)",
              fontsize=8, color=color)
    plt.axis('off')

plt.suptitle('Green = Correct | Red = Wrong', fontsize=13)
plt.tight_layout()
plt.show()